In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
from urllib.request import urlretrieve

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
urlretrieve(url, "course_lead_scoring_2026.csv")

('course_lead_scoring_2026.csv', <http.client.HTTPMessage at 0x2db24c1a0d0>)

In [5]:
import numpy as np
import pandas as pd

In [6]:
df = pd.read_csv("course_lead_scoring_2026.csv")
categorical_columns = ["lead_source", "industry", "employment_status", "location"] 
Numerical_columns = ["annual_income", "number_of_courses_viewed",
                  "interaction_count", "lead_score"]

In [7]:
for c in categorical_columns:
    df[c] = df[c].fillna("NA")
for c in Numerical_columns:
    df[c] = df[c].fillna(0.0)

print(df["industry"].value_counts())
print("Most frequent industry:", df["industry"].mode()[0])

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64
Most frequent industry: technology


In [8]:
y = df["converted"]
X = df.drop(columns=["converted"])

In [9]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


Q2. Correlation Matrix
Create the [correlation matrix](https://www.google.com/search?q=correlation+matrix) for the numerical features of your dataset.
In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

Only consider the pairs above when answering this question.

### Split the data

- Split the data with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
```

- Make sure that the target value `converted` is not in your dataframe.

In [10]:
print(df[Numerical_columns].corr())

                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  


In [11]:
print(df["interaction_count"].corr(df["lead_score"]))
print(df["number_of_courses_viewed"].corr(df["lead_score"]))
print(df["number_of_courses_viewed"].corr(df["interaction_count"]))
print(df["annual_income"].corr(df["interaction_count"]))

0.9157457980418667
0.757204239210489
0.7216090038937003
0.12284235135955941


In [12]:
from sklearn.model_selection import train_test_split

In [13]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42 )


In [15]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [16]:
y_train = df_train["converted"].values
y_val = df_val["converted"].values
y_test = df_test["converted"].values

In [17]:
del df_train["converted"]
del df_val["converted"]
del df_test["converted"]

In [18]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37
1,events,finance,employed,south_america,71662.0,0,0,0.09
2,referral,NA,employed,asia,40220.0,2,5,0.55
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45
4,events,technology,employed,north_america,89316.0,1,5,0.00


### Question 3

- Calculate the mutual information score between `converted` and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using `round(score, 2)`.

Which of these variables has the biggest mutual information score?

- `industry`
- `location`
- `lead_source`
- `employment_status`

In [19]:
from sklearn.metrics import mutual_info_score
for c in categorical_columns:
    score = mutual_info_score(df_train[c], y_train)
    print(c, round(score, 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
  - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
  - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.55
- 0.65
- 0.75
- 0.85

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

features = Numerical_columns + categorical_columns
dv = DictVectorizer(sparse=False)

train_dicts = df_train[features].to_dict(orient="records")
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[features].to_dict(orient="records")
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_val)
accuracy = accuracy_score(y_val, predictions)



print("Accuracy:", accuracy)
print("Rounded:", round(accuracy, 2))


Accuracy: 0.645
Rounded: 0.65


- Let's find the least useful feature using the _feature elimination_ technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

> **Note**: The difference doesn't have to be positive.

In [24]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

text_columns = ["lead_source", "industry", "employment_status", "location"]
number_columns = ["annual_income", "number_of_courses_viewed",
                  "interaction_count", "lead_score"]
all_features = text_columns + number_columns


def get_accuracy(features):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient="records"))
    X_val = dv.transform(df_val[features].to_dict(orient="records"))

    model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))



original = get_accuracy(all_features)
print("Original accuracy:", original)


for feature in ["lead_source", "number_of_courses_viewed", "interaction_count"]:
    features_without = [f for f in all_features if f != feature]
    new_accuracy = get_accuracy(features_without)
    difference = original - new_accuracy
    print(feature, "| accuracy:", new_accuracy, "| difference:", difference)

Original accuracy: 0.645
lead_source | accuracy: 0.642 | difference: 0.0030000000000000027
number_of_courses_viewed | accuracy: 0.643 | difference: 0.0020000000000000018
interaction_count | accuracy: 0.601 | difference: 0.04400000000000004


In [25]:
results = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    accuracy = accuracy_score(y_val, model.predict(X_val))
    results[C] = round(accuracy, 3)       # round to 3 decimals
    print("C =", C, "| accuracy =", results[C])

# Best accuracy; on a tie, take the smallest C
best_accuracy = max(results.values())
best_C = min(C for C in results if results[C] == best_accuracy)
print("Best C:", best_C)

C = 1e-06 | accuracy = 0.598
C = 1e-05 | accuracy = 0.598
C = 0.0001 | accuracy = 0.613
C = 0.001 | accuracy = 0.645
Best C: 0.001
